<a href="https://colab.research.google.com/github/blp-hue/deteccao-de-anomalias-de-transacoes-em-python/blob/main/deteccao_de_anomalias_em_transacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(url)
df.head()
df.info()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
nulos = df.isnull().sum()

duplicatas = df.duplicated().sum()
df_clean = df.drop_duplicates()

df_clean = df_clean.copy()
df_clean['Class'] = df_clean['Class'].astype(int)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_clean['Amount_scaled'] = scaler.fit_transform(df_clean[['Amount']])

df_clean['Class_label'] = df_clean['Class'].map({0: 'Legítima', 1: 'Fraude'})

In [ ]:
df_clean.describe().T

contagem = df_clean['Class_label'].value_counts()
pct      = df_clean['Class_label'].value_counts(normalize=True) * 100
display(pd.DataFrame({'Quantidade': contagem, 'Percentual (%)': pct.round(4)}))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.pipeline import Pipeline
import warnings

FEATURES = [col for col in df_clean.columns
            if col.startswith('V') or col in ['Amount_scaled', 'Time_hours']]
TARGET = 'Class'

X = df_clean[FEATURES]
y = df_clean[TARGET]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # penaliza mais os erros na classe minoritária (fraude)
    solver='lbfgs',
    random_state=42
)

modelo.fit(X_train, y_train)

y_prob = modelo.predict_proba(X_test)[:, 1]   # P(fraude)
y_pred = modelo.predict(X_test)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

metricas_modelo = {
    'Métrica'  : ['Accuracy', 'Precision (Fraude)', 'Recall (Fraude)',
                  'F1-Score (Fraude)', 'AUC-ROC'],
    'Valor'    : [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        auc(*roc_curve(y_test, y_prob)[:2])
    ],
    'Interpretação': [
        'Acertos gerais do modelo',
        'Das alertas de fraude, quantas são reais',
        'Das fraudes reais, quantas foram detectadas  ← mais crítico',
        'Equilíbrio entre Precision e Recall',
        'Área sob a curva ROC (1.0 = perfeito)'
    ]
}
display(pd.DataFrame(metricas_modelo).set_index('Métrica'))


In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling  import SMOTE
from collections import Counter

# Separação clara entre fraudes e normais antes do balanceamento
fraudes = df_clean[df_clean['Class'] == 1]
normais = df_clean[df_clean['Class'] == 0]

In [ ]:
ref = RandomUnderSampler(random_state=42)
X_under, y_under = ref.fit_resample(X, y)

contagem_under = Counter(y_under)

df_under = pd.DataFrame(X_under, columns=FEATURES)
df_under['Class'] = y_under.values
df_under['Class_label'] = df_under['Class'].map({0: 'Normal', 1: 'Fraude'})

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_over, y_over = smote.fit_resample(X, y)

contagem_over = Counter(y_over)

df_over = pd.DataFrame(X_over, columns=FEATURES)
df_over['Class'] = y_over.values
df_over['Class_label'] = df_over['Class'].map({0: 'Normal', 1: 'Fraude'})

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    max_depth    = 10,
    class_weight = 'balanced',
    n_estimators = 50,
    random_state = 42,
    n_jobs       = -1        # usa todos os núcleos disponíveis
)

rf.fit(X_train, y_train)

y_pred_ref = ref.predict (X_test)